## Experiment 3: Data Cleaning and Preparation

**Aim:** To perform data cleaning and preparation on a telecom customer churn dataset to gain insights into the factors that contribute to customer churn.

In [1]:
# A52 Dipanshu Ambilkar
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
np.random.seed(2)
n = 300
df = pd.DataFrame({
    "customer_id": range(1, n + 1),
    "gender": np.random.choice(["Male", "Female", "male ", "FEMALE"], n),
    "age": np.random.randint(18, 70, n),
    "tenure": np.random.randint(1, 72, n),
    "monthly_charges": np.round(np.random.uniform(20, 120, n), 2),
    "total_charges": np.round(np.random.uniform(100, 8000, n), 2),
    "contract": np.random.choice(["Month-to-month", "One year", "Two year"], n),
    "churn": np.random.choice(["Yes", "No"], n),
})
df.head()

,customer_id,gender,age,tenure,monthly_charges,total_charges,contract,churn
0,1,Male,51,21,117.92,6825.09,Two year,No
1,2,FEMALE,56,28,28.16,6095.94,One year,Yes
2,3,Female,25,32,55.84,3074.01,Two year,No
3,4,Male,50,10,108.77,4773.68,Month-to-month,Yes
4,5,male,67,21,82.40,2290.87,Two year,No


In [3]:
df.loc[np.random.choice(n, 15), "age"] = np.nan
df.loc[np.random.choice(n, 10), "monthly_charges"] = np.nan
df.loc[np.random.choice(n, 5), "monthly_charges"] = 9999
df = pd.concat([df, df.iloc[:8]], ignore_index=True)
print("shape after adding missing values, outliers and duplicates:", df.shape)

shape after adding missing values, outliers and duplicates: (308, 8)


In [4]:
print("missing values:")
print(df.isnull().sum())
print("duplicate rows:", df.duplicated().sum())

missing values:
customer_id         0
gender              0
age                15
tenure              0
monthly_charges     8
total_charges       0
contract            0
churn               0
dtype: int64
duplicate rows: 8


In [5]:
df = df.drop_duplicates()
print("rows after removing duplicates:", len(df))

rows after removing duplicates: 300


In [6]:
df["age"] = df["age"].fillna(df["age"].median())
print("missing age now:", df["age"].isnull().sum())

missing age now: 0


In [7]:
df["monthly_charges"] = df["monthly_charges"].fillna(df["monthly_charges"].median())
print("missing monthly_charges now:", df["monthly_charges"].isnull().sum())

missing monthly_charges now: 0


In [8]:
df["gender"] = df["gender"].str.strip().str.lower()
df["age"] = df["age"].astype(int)
print("gender values now:", list(df["gender"].unique()))

gender values now: ['male', 'female']


In [9]:
q1 = df["monthly_charges"].quantile(0.25)
q3 = df["monthly_charges"].quantile(0.75)
iqr = q3 - q1
df = df[(df["monthly_charges"] >= q1 - 1.5 * iqr) & (df["monthly_charges"] <= q3 + 1.5 * iqr)]
print("rows after outlier removal:", len(df))

rows after outlier removal: 295


In [10]:
df["avg_monthly"] = df["total_charges"] / df["tenure"]
df["is_senior"] = (df["age"] >= 60).astype(int)
df.head()

,customer_id,gender,age,tenure,monthly_charges,total_charges,contract,churn,avg_monthly,is_senior
0,1,male,51,21,117.92,6825.09,Two year,No,325.004286,0
1,2,female,56,28,28.16,6095.94,One year,Yes,217.712143,0
2,3,female,25,32,55.84,3074.01,Two year,No,96.062813,0
3,4,male,50,10,108.77,4773.68,Month-to-month,Yes,477.368000,0
4,5,male,67,21,82.40,2290.87,Two year,No,109.089048,1


In [11]:
num = ["age", "tenure", "monthly_charges", "total_charges", "avg_monthly"]
df[num] = StandardScaler().fit_transform(df[num])
df.head()

,customer_id,gender,age,tenure,monthly_charges,total_charges,contract,churn,avg_monthly,is_senior
0,1,male,0.396141,-0.645453,1.785469,1.143496,Two year,No,0.100446,0
1,2,female,0.742296,-0.301134,-1.446509,0.821006,One year,Yes,-0.119779,0
2,3,female,-1.403861,-0.104380,-0.449838,-0.515539,Two year,No,-0.369472,0
3,4,male,0.326911,-1.186527,1.456006,0.236195,Month-to-month,Yes,0.413182,0
4,5,male,1.503835,-0.645453,0.506505,-0.861908,Two year,No,-0.342735,1


In [12]:
df = pd.get_dummies(df, columns=["gender", "contract"], drop_first=True)
print("shape after encoding:", df.shape)

shape after encoding: (295, 11)


In [13]:
train, test = train_test_split(df, test_size=0.2, random_state=2)
print("train shape:", train.shape, " test shape:", test.shape)

train shape: (236, 11)  test shape: (59, 11)


In [14]:
train.to_csv("churn_cleaned.csv", index=False)
print("done")

done
